## Setup

In [11]:
import os
import numpy as np

import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torchvision.io import decode_image

In [12]:
from google.colab import drive
drive.mount('/content/drive')

base_dir = "/content/drive/MyDrive/Lucas"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## NIL embeddings (UniCAS)

In [13]:
emb_path = base_dir + "/Embeddings/NIL/"
nil_embeddings = []
for i in range(1,201):
  nil_embeddings.append(torch.load(emb_path + f"{i}.pt"))

In [14]:
nil_embeddings[0].shape

torch.Size([45, 1024])

## Carregando ambil.pth (treinado sobre o BMT, com UniCAS como backbone)

In [15]:
'''
Feito com auxílio do  Claude Code (Opus 5.5 / High)
'''

import torch.nn as nn

class GatedAttentionMIL(nn.Module):
    """ABMIL com atenção com porta (Ilse et al., 2018)."""

    def __init__(self, topk=8, in_dim=1024, hid=256, att_hid=128, n_classes=3, p_drop=0.25):
        super().__init__()

        self.topk = topk

        # projeção: reduz 1024 → 256 antes da atenção
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hid),
            nn.ReLU(),
            nn.Dropout(p_drop),
        )
        # dois ramos da atenção com porta
        self.att_V = nn.Sequential(nn.Linear(hid, att_hid), nn.Tanh())
        self.att_U = nn.Sequential(nn.Linear(hid, att_hid), nn.Sigmoid())
        self.att_w = nn.Linear(att_hid, 1)

        self.classifier = nn.Linear(hid, n_classes)

    def forward(self, bags):                    # bags: [B, N, 1024]
        h = self.proj(bags)                     # [B, N, 256]

        a = self.att_w(self.att_V(h) * self.att_U(h))   # [B, N, 1]

        if self.topk == -1:
          self.topk = h.shape[1]

        _, topk_idx = torch.topk(a,self.topk,dim=1)
        topk_idx = torch.flatten(topk_idx)

        a = a[:,topk_idx]
        a = torch.softmax(a, dim=1)                     # pesos somam 1 no bag

        z = (a * h[:,topk_idx]).sum(dim=1)              # [B, 256] — bag embedding

        # logits [B,3] // indices topk tiles // attention scores (0,1) // logits de cada tile do topk
        return self.classifier(z), topk_idx, a, self.classifier(h[:,topk_idx])

## Variando topk e calculando acertos

In [16]:
topks = [1,2,4,8,16,32,-1]

labels = {0: 'NIL', 1: 'LSIL', 2: 'HSIL'}
for topk in topks:
  device = "cuda" if torch.cuda.is_available() else "cpu"
  abmil = GatedAttentionMIL(topk=topk).to(device)
  state = torch.load(base_dir + "/abmil.pth", map_location='cpu')
  abmil.load_state_dict(state)

  for param in abmil.parameters():
      param.requires_grad = False

  abmil.eval()
  nil = 0
  with torch.no_grad():
    for i in range(200):
      X = nil_embeddings[i].unsqueeze(0)
      X = X.to(device)

      y,a,att,logits_topk = abmil(X)
      if labels[y.argmax().item()] == "NIL":
        nil += 1

  print(f"topk={topk}: {nil}/200 acertos")

topk=1: 93/200 acertos
topk=2: 111/200 acertos
topk=4: 141/200 acertos
topk=8: 156/200 acertos
topk=16: 173/200 acertos
topk=32: 190/200 acertos
topk=-1: 194/200 acertos
